In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

# 2. LOAD DATASET
file_path = "/content/adult.csv"

# CHANGE THIS TO YOUR TARGET COLUMN
target_column = "income"

df = pd.read_csv(file_path)

print("First 5 Rows:")
print(df.head())

print("\nDataset Shape:", df.shape)

# 3. DATA PREPROCESSING
df.columns = df.columns.astype(str).str.strip()

if target_column not in df.columns:
    raise ValueError(
        f"Target column not found. Available columns: {df.columns.tolist()}"
    )

# Remove duplicates and missing target values
df = df.drop_duplicates()
df = df.dropna(subset=[target_column])

# Separate features and target
X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()

# Remove rows with missing target values
# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y.astype(str))

print("\nTarget Classes:", label_encoder.classes_)
print("Class Distribution:")
print(pd.Series(y).value_counts())

if len(np.unique(y)) < 2:
    raise ValueError("Target must contain at least two classes.")

# 4. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("\nNumerical Features:", numeric_features)
print("Categorical Features:", categorical_features)

# 5. HANDLE MISSING VALUES AND ENCODE CATEGORICAL DATA
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 6. SPLIT DATASET
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

# 7. CREATE RANDOM FOREST MODEL
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        criterion="gini",
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

# 8. TRAIN MODEL
model.fit(X_train, y_train)

print("\nRandom Forest Model Trained Successfully!")

# 9. MAKE PREDICTIONS
y_pred = model.predict(X_test)

# 10. MODEL EVALUATION
print("\nMODEL EVALUATION")

print("Accuracy:",
      round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

print("Precision:",
      round(precision_score(y_test, y_pred, average="weighted", zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, y_pred, average="weighted", zero_division=0), 4))

print("F1 Score:",
      round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 4))

# 11. CONFUSION MATRIX
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(label_encoder.classes_))
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

# 12. FEATURE IMPORTANCE
rf_model = model.named_steps["classifier"]

feature_names = model.named_steps[
    "preprocessing"
].get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance.head(15).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

# 13. TRAINING AND TESTING ACCURACY
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print("\nTraining Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy:", round(test_accuracy, 4))

print("\nRANDOM FOREST EXPERIMENT COMPLETED!")


First 5 Rows:
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18 